# Calling a minimal DIRAC setup to obtain matrix elements and Basis definitions

In [1]:
from ase import Atoms
from ase.io import read
from ase.units import Ha
from ase_dirac import DIRAC
from ase.calculators.calculator import CalculationFailed
import h5py
import numpy as np
from typing import Optional, Union, Literal
import matplotlib.pyplot as plt

In [2]:
xyz_string = """2
F2
H      0 0 0
H      0.00000      0.0     -1.0
"""

inp_str = """ 
**DIRAC
.TITLE
Molecular Te. DEA from the dianion.
.WAVE F
**HAMILTONIAN
.LVCORR
**WAVE FUNCTIONS
.SCF
*SCF
.MAXITR
1
**MOLECULE
*BASIS
.DEFAULT
dyall.3zp
.Nosym
*END OF
"""

with open("F2.xyz", "w") as f:
    f.write(xyz_string)

with open("nonrel_scf.inp", "w") as f:
    f.write(inp_str)

In [3]:
ase_molecule = read("F2.xyz", format="xyz")



In [4]:
ase_molecule.calc = DIRAC.from_file("nonrel_scf.inp")

In [5]:
try:
    ase_molecule.get_potential_energy() / Ha
except CalculationFailed: 
    print("\n\nCalculation failed because we requested a single SCF iteration so it is obviously not converged.")
    is_there_h5 = ase_molecule.calc.h5_exists
    print(f'However, h5 file is ready: {is_there_h5}')

 starting DIRAC for molecule dirac.xyz with input dirac.inp
  creating archive file  dirac.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : ABNORMAL (CHECK DIRAC OUTPUT)


Calculation failed because we requested a single SCF iteration so it is obviously not converged.
However, h5 file is ready: True


In [6]:
data_L = ase_molecule.calc.extract_h5_value("input/aobasis/1/")


data_S = ase_molecule.calc.extract_h5_value("input/aobasis/2/")
one_electron_ints_symm = ase_molecule.calc.extract_h5_value("result/operators/ao_matrices")



In [7]:
def extract_basis_data(Dirac_Calculator: DIRAC, component: Literal['Large', 'Small']='Large'):
    if not Dirac_Calculator.h5_exists:
        raise FileNotFoundError('There is no availiable h5 file')
    
    basis_number = '1' if component == 'Large' else '2'

    R_array = Dirac_Calculator.extract_h5_value(f'input/aobasis/{basis_number}/center').reshape(-1,3)
    exps_array = Dirac_Calculator.extract_h5_value(f'input/aobasis/{basis_number}/exponents')
    l_array = Dirac_Calculator.extract_h5_value(f'input/aobasis/{basis_number}/orbmom')

    print(R_array)
    print(exps_array)
    print(l_array)
    


"""
Dataset: input/aobasis/1/angular (1,)
Dataset: input/aobasis/1/aobasis_id (1,)
Dataset: input/aobasis/1/center (33,)
Dataset: input/aobasis/1/contractions (11,)
Dataset: input/aobasis/1/exponents (11,)
Dataset: input/aobasis/1/n_ao (1,)
Dataset: input/aobasis/1/n_cont (11,)
Dataset: input/aobasis/1/n_prim (11,)
Dataset: input/aobasis/1/n_shells (1,)
Dataset: input/aobasis/1/orbmom (11,)
"""
    


'\nDataset: input/aobasis/1/angular (1,)\nDataset: input/aobasis/1/aobasis_id (1,)\nDataset: input/aobasis/1/center (33,)\nDataset: input/aobasis/1/contractions (11,)\nDataset: input/aobasis/1/exponents (11,)\nDataset: input/aobasis/1/n_ao (1,)\nDataset: input/aobasis/1/n_cont (11,)\nDataset: input/aobasis/1/n_prim (11,)\nDataset: input/aobasis/1/n_shells (1,)\nDataset: input/aobasis/1/orbmom (11,)\n'

In [8]:
extract_basis_data(ase_molecule.calc)

[[0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]
 [0.         0.         0.94486306]]
[9.01657212e+02 1.34649031e+02 3.05473364e+01 8.62453493e+00
 2.80494579e+00 1.00896460e+00 3.91394822e-01 1.60426029e-01
 6.71243724e-02 1.23551545e+00 2.93467565e-01]
[1 1 1 1 1 1 1 1 1 2 2]


There might be a bug here.
